In [1]:
!pip install nltk sastrawi tensorflow pandas scikit-learn


In [2]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense


In [3]:
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
df = pd.read_csv("dataset_playstore.csv")
df.head()


,review,rating
0,shopee sangat membantu sekali buat saya,5
1,okk pelayannya,5
2,bagus bngt,5
3,shopee is the best...,5
4,aplikasi ini sangat mudah dan cepat,5


In [5]:
stop_words = set(stopwords.words('indonesian'))
stemmer = StemmerFactory().create_stemmer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = text.strip()
    text = " ".join([w for w in text.split() if w not in stop_words])
    return stemmer.stem(text)

df['clean_review'] = df['review'].apply(preprocess_text)


In [6]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df['clean_review'])

X = tokenizer.texts_to_sequences(df['clean_review'])
X = pad_sequences(X, maxlen=100)


In [7]:
le = LabelEncoder()
y = le.fit_transform(df['label'])


KeyError: 'label'

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
model = Sequential([
    Embedding(5000, 64, input_length=100),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)


In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("Akurasi testing:", acc)


In [ ]:
sample = ["aplikasinya jelek dan sering error"]
seq = tokenizer.texts_to_sequences(sample)
pad = pad_sequences(seq, maxlen=100)

pred = model.predict(pad)
print("Prediksi sentimen:", le.inverse_transform([int(pred > 0.5)]))
